In [2]:
from datasets import load_dataset
import torch
import numpy as np
import chess
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [3]:
dataset = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train",
    cache_dir="./training_data"
)

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [4]:
split1 = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = split1["train"]
temp_ds  = split1["test"]

# second split: val vs test
split2 = temp_ds.train_test_split(test_size=0.5, seed=42)

val_ds   = split2["train"]
test_ds  = split2["test"]

In [5]:
#see https://official-stockfish.github.io/docs/nnue-pytorch-wiki/docs/nnue.html#halfkp
PIECE_MAP = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
}

# using the same formula provided by the stockfish site
def halfkp_index(piece, piece_square, king_square):
    piece_type = PIECE_MAP[piece.piece_type]
    piece_color = 0 if piece.color == chess.WHITE else 1
    p_idx = piece_type * 2 + piece_color
    return piece_square + (p_idx + king_square * 10) * 64

def extract_halfkp(board, perspective):
    king_sq = board.king(perspective)
    feats = []

    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue
        feats.append(halfkp_index(piece, sq, king_sq))

    return feats

In [6]:
class ChessDataset(Dataset):
    def __init__(self, data, max_samples=None):
        self.ds = data if max_samples is None else data.select(range(max_samples))

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        board = chess.Board(item["fen"])

        white_feats = extract_halfkp(board, chess.WHITE)
        black_feats = extract_halfkp(board, chess.BLACK)

        # side to move
        stm = 1.0 if board.turn == chess.WHITE else 0.0

        # clip and normalize bc centipawn can go into the infinities as outliers (for mate)
        cp = item["cp"] if item["cp"] is not None else 0
        cp = max(min(cp, 1000), -1000) / 1000.0 

        return {
            "white": torch.tensor(white_feats, dtype=torch.long),
            "black": torch.tensor(black_feats, dtype=torch.long),
            "stm": torch.tensor(stm, dtype=torch.float32),
            "target": torch.tensor([cp], dtype=torch.float32)
        }

In [7]:
def collate_fn(batch):
    white = [b["white"] for b in batch]
    black = [b["black"] for b in batch]

    stm = torch.stack([b["stm"] for b in batch])
    target = torch.stack([b["target"] for b in batch])

    return white, black, stm, target

In [8]:
NUM_FEATURES = 40960
EMB_DIM = 2048

class NNUE(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(NUM_FEATURES, EMB_DIM)

        self.mlp = nn.Sequential(
            nn.Linear(2 * EMB_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, feats):
        padded = torch.nn.utils.rnn.pad_sequence(
            feats, batch_first=True, padding_value=0
        )
        emb = self.embedding(padded)
        mask = (padded != 0).unsqueeze(-1)
        return (emb * mask).sum(dim=1)

    
    def forward(self, white, black, stm):
        w = self.encode(white)
        b = self.encode(black)

        x = torch.where(
            stm.unsqueeze(1) == 1,
            torch.cat([w, b], dim=1),
            torch.cat([b, w], dim=1),
        )

        return self.mlp(x)

In [9]:
train_data = ChessDataset(train_ds, max_samples=100000)
val_data   = ChessDataset(val_ds, max_samples=20000)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [10]:
model = NNUE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [11]:
if torch.cuda.is_available():
    device = torch.device('cuda')
#elif torch.backends.mps.is_available():
#    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [13]:
EPOCHS = 10

for epoch in range(EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")

    for white, black, stm, target in train_bar:
        stm = stm.to(device)
        target = target.to(device)

        white = [w.to(device) for w in white]
        black = [b.to(device) for b in black]

        pred = model(white, black, stm)
        loss = loss_fn(pred, target)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        train_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    val_loss = 0.0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [VAL]")

    with torch.no_grad():
        for white, black, stm, target in val_bar:
            stm = stm.to(device)
            target = target.to(device)

            white = [w.to(device) for w in white]
            black = [b.to(device) for b in black]

            pred = model(white, black, stm)
            loss = loss_fn(pred, target)

            val_loss += loss.item()
            val_bar.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    print(
        f"\nEpoch {epoch+1} Summary | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}\n"
    )

Epoch 1 [VAL]: 100%|██████████| 625/625 [00:05<00:00, 108.25it/s, loss=0.0826]



Epoch 1 Summary | Train Loss: 0.078964 | Val Loss: 0.100484



Epoch 2 [VAL]: 100%|██████████| 625/625 [00:06<00:00, 102.06it/s, loss=0.101] 



Epoch 2 Summary | Train Loss: 0.063602 | Val Loss: 0.108098



Epoch 3 [VAL]: 100%|██████████| 625/625 [00:05<00:00, 111.05it/s, loss=0.083] 


...


Epoch 8 Summary | Train Loss: 0.063774 | Val Loss: 0.102593



Epoch 9 [VAL]: 100%|██████████| 625/625 [00:05<00:00, 108.50it/s, loss=0.112] 



Epoch 9 Summary | Train Loss: 0.066176 | Val Loss: 0.120353



Epoch 10 [VAL]: 100%|██████████| 625/625 [00:05<00:00, 108.71it/s, loss=0.089] 


Epoch 10 Summary | Train Loss: 0.060359 | Val Loss: 0.117786



In [14]:
test_data  = ChessDataset(test_ds, max_samples=100000)
test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

model.eval()

test_loss = 0.0
mae = 0.0
count = 0

with torch.no_grad():
    for white, black, stm, target in test_loader:
        stm = stm.to(device)
        target = target.to(device)

        white = [w.to(device) for w in white]
        black = [b.to(device) for b in black]

        pred = model(white, black, stm)

        loss = loss_fn(pred, target)
        test_loss += loss.item()

        # optional: MAE (more interpretable than MSE)
        mae += torch.abs(pred - target).sum().item()
        count += target.numel()

test_loss /= len(test_loader)
mae /= count

print(f"\nFINAL TEST RESULTS")
print(f"Test MSE: {test_loss:.6f}")
print(f"Test MAE: {mae:.6f}")


FINAL TEST RESULTS
Test MSE: 0.111956
Test MAE: 0.216219


In [15]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch,
    "loss": train_loss
}, "nnue_checkpoint.pt")